# Cubism on a cube tessellation: Obama

Tristan Miller's Cube Tessellation, folded from an 8 × 8 grid, painted as a cubist portrait of Barack Obama in the manner of a David Hockney photo joiner.

**The geometry.** Each cube shows three faces. Tilt the model until you look straight along one cube axis, about 55° off the paper's normal, and two of the three faces turn edge-on and vanish, while the third shows as true squares. So every visible bit of paper belongs to exactly one axis.

**The piece.** Head-on, the 25 faces add up to an oval head: hair near the top, eyes and nose in the middle, mouth and chin below, against Kehinde Wiley-inspired leaves and flowers (chrysanthemums, jasmine, agapanthus). Each face is its own view of a 3D scan of Obama, from its own angle, zoom and turn, so tilting toward any axis breaks the head into a jumble of that axis's views. The faces were painted as pointillist oil in [`obama_paint.ipynb`](obama_paint.ipynb); how they were made is in [`obama_faces/README.md`](obama_faces/README.md).

**This demo is virtual:** no paper, no visibility map, and the finished model's 25 faces as ideal squares (step 18 of the diagrams), leaving out the flat triangles at its rim. No GPU is needed.

The guitar version of this notebook, and the provenance explorer that goes with it, are at commit `acd474b` on `main`.

## Setup

The geometry and renderer. No GPU needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

# The finished model's faces (step 18 of the diagrams). Start from cubes at lattice points of the
# plane x + y + z = 0: head-on, along (1, 1, 1), they tile as tumbling blocks, each showing its +x, +y
# and +z faces. Nine cubes give rows of 1, 2, 3, 2, 1. The model differs at the bottom and the sides:
# - The bottom is a hollow corner, not a cube: drop the bottom cube's two side faces, and the outer
#   side faces of the two cubes above it.
# - Each side has a wall rising from the outer cube's top to the second row.
CUBES = [(a, b, -(a + b)) for a in range(3) for b in range(3)]
DROP = {(0, (3, 2, -4)), (1, (2, 3, -4)), (0, (3, 1, -3)), (1, (1, 3, -3))}   # (axis, corner)
WALLS = [(1, (2, 0, -1)), (0, (0, 2, -1))]                  # (axis, corner)
AXES = ("x", "y", "z")
E = np.eye(3)
HEAD_ON = np.ones(3) / np.sqrt(3)
HEAD_ON_UP = np.array([-1.0, -1.0, 2.0]) / np.sqrt(6)      # screen up when seen head-on
LIT = (0.8, 0.9, 1.0)                                      # x, y, z faces lit from above and to the right


def face(k, corner):
    i, j = [m for m in range(3) if m != k]
    return (k, np.array(corner, float), E[i], E[j])


def faces():
    """Every visible face: (axis it faces, corner, edge 1, edge 2)."""
    out = [face(k, np.add(p, E[k])) for p in CUBES for k in range(3)]
    out = [f for f in out if (f[0], tuple(f[1].astype(int))) not in DROP]
    return out + [face(k, c) for k, c in WALLS]


def camera(t=0.0, axis=2):
    """Toward-viewer direction and screen right and up, tilted a fraction t of the way from head-on to
    looking straight along an axis. Up stays as close to head-on's up as it can, as when tilting a
    model in your hands."""
    a, b = HEAD_ON, E[axis]
    ang = np.arccos(a @ b)
    d = (np.sin((1 - t) * ang) * a + np.sin(t * ang) * b) / np.sin(ang)
    up = HEAD_ON_UP - (HEAD_ON_UP @ d) * d
    up /= np.linalg.norm(up)
    return d, np.cross(up, d), up


def bounds(cam, fs):
    _, r, u = cam
    pts = np.array([c + s * e1 + q * e2 for _, c, e1, e2 in fs for s in (0, 1) for q in (0, 1)])
    return (pts @ r).min(), (pts @ r).max(), (pts @ u).min(), (pts @ u).max()


def render(pictures=None, t=0.0, axis=2, size=512, margin=0.06, bg=1.0, light=None, textures=None):
    """The model seen at a tilt t toward an axis. pictures: one (H, W, 3) array in [0, 1] per axis,
    each covering (center-cropped to) the box around that axis's faces as seen straight along it.
    light: a brightness per axis, to see the faces as lit paper, e.g. LIT.
    textures: instead of pictures, one (n, n, 3) array per face, rows along its edge 2."""
    fs = faces()
    d, r, u = cam = camera(t, axis)
    x0, x1, y0, y1 = bounds(cam, fs)
    span = max(x1 - x0, y1 - y0) * (1 + 2 * margin)
    X, Y = np.meshgrid((x0 + x1) / 2 + ((np.arange(size) + 0.5) / size - 0.5) * span,
                       (y0 + y1) / 2 - ((np.arange(size) + 0.5) / size - 0.5) * span)
    out = np.full((size, size, 3), bg)
    depth = np.full((size, size), -np.inf)
    cams = [camera(1.0, k) for k in range(3)]
    boxes = [bounds(cams[k], [f for f in fs if f[0] == k]) for k in range(3)]
    for i, (k, c, e1, e2) in enumerate(fs):
        if E[k] @ d < 1e-6:                                # seen edge-on
            continue
        inv = np.linalg.inv(np.array([[e1 @ r, e2 @ r], [e1 @ u, e2 @ u]]))
        qx, qy = X - c @ r, Y - c @ u
        s, q = inv[0, 0] * qx + inv[0, 1] * qy, inv[1, 0] * qx + inv[1, 1] * qy
        P = c + s[..., None] * e1 + q[..., None] * e2
        z = P @ d
        hit = (s >= 0) & (s <= 1) & (q >= 0) & (q <= 1) & (z > depth)
        if textures is not None:
            n = textures[i].shape[0]
            si, qi = (s * n).astype(int).clip(0, n - 1), (q * n).astype(int).clip(0, n - 1)
            out[hit] = textures[i][qi[hit], si[hit]] * (1 if light is None else light[k])
            depth[hit] = z[hit]
            continue
        # Each face shows the part of its axis's picture that it covers when seen along that axis.
        _, rk, uk = cams[k]
        bx0, bx1, by0, by1 = boxes[k]
        pic = pictures[k]
        H, W = pic.shape[:2]
        scale = max(W / (bx1 - bx0), H / (by1 - by0))        # cover the box, cropping the overflow
        px = ((P @ rk - (bx0 + bx1) / 2) * scale + W / 2).astype(int).clip(0, W - 1)
        py = (((by0 + by1) / 2 - P @ uk) * scale + H / 2).astype(int).clip(0, H - 1)
        out[hit] = pic[py[hit], px[hit]] * (1 if light is None else light[k])
        depth[hit] = z[hit]
    return out


def placeholder(text, color, size=256):
    """A stand-in picture: a letter on stripes, to check the geometry without a GPU."""
    img = Image.new("RGB", (size, size), color)
    draw = ImageDraw.Draw(img)
    for i in range(-size, size, 24):
        draw.line([(i, 0), (i + size, size)], fill=tuple(int(v * 0.8) for v in color), width=6)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", size // 3)
    except OSError:
        font = ImageFont.load_default(size // 3)
    draw.text((size / 2, size / 2), text, fill="black", font=font, anchor="mm")
    return np.asarray(img, float) / 255


def plot_tilts(pictures, titles, tilts=(0, 0.25, 0.5, 0.75, 1), size=256, light=None, textures=None):
    """One row per axis: the model tilted from head-on to looking straight along that axis."""
    fig, axes = plt.subplots(3, len(tilts), figsize=(3 * len(tilts), 9.3))
    for k in range(3):
        for j, t in enumerate(tilts):
            axes[k, j].imshow(render(pictures, t, k, size, light=light, textures=textures))
            axes[k, j].set_title("head-on" if t == 0 else f"{titles[k]}" if t == 1 else f"{t:.0%} toward {AXES[k]}",
                                 fontsize=10)
            axes[k, j].axis("off")
    fig.tight_layout()
    return fig


def save_gif(pictures, path, frames=14, size=360, hold=6, light=None, textures=None):
    """Head-on, tilt to x and back, then y, then z."""
    ease = lambda s: (1 - np.cos(np.pi * s)) / 2
    imgs = []
    for k in range(3):
        ts = [ease(i / frames) for i in range(frames + 1)]
        seq = [0.0] * 2 + ts + [1.0] * hold + ts[::-1]
        imgs += [Image.fromarray((render(pictures, t, k, size, light=light, textures=textures) * 255).astype(np.uint8)) for t in seq]
    imgs[0].save(path, save_all=True, append_images=imgs[1:], duration=60, loop=0)

## The three windows

Each axis's faces form a window. Stand-in letters show the shapes and how the tilt goes: head-on (left), each cube shows a piece of all three; tilted all the way (right), the other faces have vanished edge-on.

In [ ]:
STAND_INS = [placeholder("X", (230, 120, 90)), placeholder("Y", (90, 160, 230)), placeholder("Z", (120, 200, 110))]
plot_tilts(STAND_INS, ("x", "y", "z"))
plt.show()

## The Obama faces

The 25 finished faces are fetched from the repo, numbered in `faces()` order, so no upload is needed. `FACES` picks the stage:

- `"oil"`: the finished pointillist oil faces (the 0.42 run).
- `"clean"`: the geometric faces before painting: the head cut into facets over the faceted flower backdrop.
- `"dots"`: the dot underpainting the oil faces started from.

In [ ]:
import urllib.request
from pathlib import Path

FACES = "oil"                 # "oil", "clean" or "dots"
BRANCH = "cubism/obama"       # "main" once merged
BASE = f"https://raw.githubusercontent.com/rath-hy/folded-illusions/{BRANCH}/cubism/obama_faces/"

folder = Path("obama_faces") / FACES
folder.mkdir(parents=True, exist_ok=True)
obama_tex = []
for i in range(len(faces())):
    f = folder / f"face_{i + 1:02d}.jpg"
    if not f.exists():
        urllib.request.urlretrieve(BASE + f"{FACES}/{f.name}", f)
    obama_tex.append(np.asarray(Image.open(f).convert("RGB"), np.float32) / 255)
print(len(obama_tex), "faces,", obama_tex[0].shape[0], "pixels across")

### Head-on, and each axis

Head-on (lit as paper would be, `LIT`) is the portrait. Tilted all the way to an axis, only that axis's faces remain.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
axes[0].imshow(render(textures=obama_tex, size=400, light=LIT)); axes[0].set_title("head-on")
for k in range(3):
    axes[k + 1].imshow(render(textures=obama_tex, t=1.0, axis=k, size=300, light=LIT)); axes[k + 1].set_title(f"tilted to {AXES[k]}")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(render(textures=obama_tex, size=900, light=LIT)); ax.set_title("head-on"); ax.axis("off")
plt.show()

### Tilting

One row per axis, from head-on to looking straight along it. Then the GIF: head-on, tilt to x and back, then y, then z. Saved as `cubism_obama.gif`.

In [ ]:
plot_tilts(None, ("x", "y", "z"), light=LIT, textures=obama_tex)
plt.show()

from IPython.display import Image as ShowImage
save_gif(None, "cubism_obama.gif", light=LIT, textures=obama_tex)
ShowImage(filename="cubism_obama.gif")

## What this leaves out

- **The real visibility map.** The demo shows ideal cube faces. The real model's faces come from patches of the folded sheet, and its rim has flat triangles, so we need the map from sheet to faces before printing. The label sheet in [`fold_map/`](fold_map/) is for finding it.
- **Viewing at about 55°.** That's steep, and real cubes are rounded and imperfect, so the squares won't line up exactly.
- **Credits.** The head is "BARACK OBAMA" by louis on Sketchfab, CC BY 4.0 (see [`obama_kit/README.md`](obama_kit/README.md)); the flower photos' credits are in [`obama_kit/flowers/README.md`](obama_kit/flowers/README.md). Kehinde Wiley's portrait gave direction only.